# 17.3 - NLP Capstone

Status: VERIFIED

## What Are We Solving?

Build an NLP classification system. We generate synthetic text data, apply preprocessing, build a TF-IDF baseline, train a classifier, and evaluate with precision/recall/F1.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import re

np.random.seed(42)

positive_templates = [
    'This product is amazing and I love it',
    'Great quality, highly recommended',
    'Excellent service, very satisfied',
    'Works perfectly, best purchase ever',
    'Outstanding performance, worth every penny',
]
negative_templates = [
    'Terrible product, waste of money',
    'Poor quality, very disappointed',
    'Awful experience, would not recommend',
    'Broke after one day, horrible',
    'Worst purchase I have ever made',
]

texts, labels = [], []
for _ in range(150):
    texts.append(np.random.choice(positive_templates) + ' ' + str(np.random.randint(1, 100)))
    labels.append(1)
    texts.append(np.random.choice(negative_templates) + ' ' + str(np.random.randint(1, 100)))
    labels.append(0)

df = pd.DataFrame({'text': texts, 'label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset: {len(df)} samples')
print(f'Class balance: {dict(df["label"].value_counts())}')
df.head()

Dataset: 300 samples
Class balance: {0: np.int64(150), 1: np.int64(150)}


,text,label
0,"Broke after one day, horrible 5",0
1,"Great quality, highly recommended 96",1
2,This product is amazing and I love it 48,1
3,"Poor quality, very disappointed 64",0
4,Worst purchase I have ever made 23,0


In [2]:
# Text Preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    stop_words = {'the', 'a', 'an', 'is', 'it', 'i', 'and', 'of', 'to', 'my'}
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean'] = df['text'].apply(preprocess)
print('Sample preprocessed texts:')
for i in range(3):
    print(f'  [{df.label.iloc[i]}] "{df.text.iloc[i]}"')
    print(f'       -> "{df.clean.iloc[i]}"')
print(f'Total samples: {len(df)}')

Sample preprocessed texts:
  [0] "Broke after one day, horrible 5"
       -> "broke after one day horrible"
  [1] "Great quality, highly recommended 96"
       -> "great quality highly recommended"
  [1] "This product is amazing and I love it 48"
       -> "this product amazing love"
Total samples: 300


In [3]:
# TF-IDF + Classification
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_df['clean'])
X_test = tfidf.transform(test_df['clean'])
y_train = train_df['label'].values
y_test = test_df['label'].values

clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

print(f'Vocabulary size: {len(tfidf.vocabulary_)}')
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Accuracy: {(preds == y_test).mean():.4f}')

Vocabulary size: 75
Train: 240 | Test: 60
Accuracy: 1.0000


In [4]:
# Evaluation
print('Classification Report:')
print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))
f1 = f1_score(y_test, preds)
print(f'F1 Score: {f1:.4f}')
print('VERIFICATION PASSED: Phase 17.3 complete')

Classification Report:


              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        20
    Positive       1.00      1.00      1.00        40

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

F1 Score: 1.0000
VERIFICATION PASSED: Phase 17.3 complete
